# System Usability Scale (SUS) — Coronary Explorer Dashboard

Hardcoded Likert responses (1–5) from **n = 3** participants. No plots — tables only, exported to `exports/` for the thesis.

**Source:** `System Usability Evaluation Answers - Spanish.csv` (survey in Spanish; standard English SUS items in tables).

**CSV exports:**
- `sus_raw_responses.csv` — raw scores per item and participant
- `sus_scores_summary.csv` — SUS sums/scores plus mean and sample std. dev.

## 1. Setup

In [13]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "src" / "blocks" / "_01_extraction.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError(
            "Could not locate repo root. Run Jupyter from the repo or from "
            "notebooks/results visualizations."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPORT_DIR = NOTEBOOK_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Exports: {EXPORT_DIR}")

Exports: C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\exports


## 2. Hardcoded raw responses (Likert 1–5)

In [14]:
SUS_ITEMS_EN = [
    "Q1: I would like to use this system frequently",
    "Q2: The system was unnecessarily complex",
    "Q3: The system was easy to use",
    "Q4: I would need technical support to use the system",
    "Q5: Functions were well integrated",
    "Q6: Too much inconsistency in the system",
    "Q7: Most people would learn quickly",
    "Q8: The system was cumbersome to use",
    "Q9: I felt confident using the system",
    "Q10: I needed to learn a lot before going",
]

SUS_RESPONSES: dict[str, list[int]] = {
    "Participant 1": [4, 1, 3, 3, 3, 3, 2, 3, 3, 3],
    "Participant 2": [5, 3, 5, 3, 4, 2, 4, 2, 3, 2],
    "Participant 3": [4, 1, 4, 1, 4, 2, 4, 1, 4, 1],
}

N_PARTICIPANTS = len(SUS_RESPONSES)
N_ITEMS = 10

raw_df = pd.DataFrame(SUS_RESPONSES, index=[f"Q{i}" for i in range(1, N_ITEMS + 1)])
raw_df.index.name = "Item"
raw_df

,Participant 1,Participant 2,Participant 3
Item,,,
Q1,4,5,4
Q2,1,3,1
Q3,3,5,4
Q4,3,3,1
Q5,3,4,4
Q6,3,2,2
Q7,2,4,4
Q8,3,2,1
Q9,3,3,4


## 3. Table 1 — export raw responses (`sus_raw_responses.csv`)

In [15]:
raw_export = raw_df.reset_index()
raw_path = EXPORT_DIR / "sus_raw_responses.csv"
raw_export.to_csv(raw_path, index=False)
print(f"Saved: {raw_path}")
raw_export

Saved: C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\exports\sus_raw_responses.csv


,Item,Participant 1,Participant 2,Participant 3
0,Q1,4,5,4
1,Q2,1,3,1
2,Q3,3,5,4
3,Q4,3,3,1
4,Q5,3,4,4
5,Q6,3,2,2
6,Q7,2,4,4
7,Q8,3,2,1
8,Q9,3,3,4
9,Q10,3,2,1


## 4. SUS scoring + Table 2 (`sus_scores_summary.csv`)

Per item: odd → `response − 1`; even → `5 − response`.  
SUS = sum of contributions × 2.5.  
Summary rows: **Mean SUS Score** (total sum 0–40 = 85; mean SUS = 70.83) and **Std. dev. (sample)**.

In [16]:
def sus_item_contribution(item_number: int, response: int) -> int:
    if item_number % 2 == 1:
        return int(response) - 1
    return 5 - int(response)


def sus_score_from_responses(responses: list[int]) -> tuple[int, float]:
    contribs = [sus_item_contribution(i + 1, r) for i, r in enumerate(responses)]
    total = sum(contribs)
    return total, float(total) * 2.5


score_rows: list[dict[str, object]] = []
for participant, responses in SUS_RESPONSES.items():
    s040, sus100 = sus_score_from_responses(responses)
    score_rows.append(
        {
            "Participant": participant,
            "Sum (0-40)": s040,
            "SUS score (0-100)": round(sus100, 2),
        }
    )

scores_df = pd.DataFrame(score_rows).set_index("Participant")
participant_sums = scores_df["Sum (0-40)"].to_numpy(dtype=float)
participant_sus = scores_df["SUS score (0-100)"].to_numpy(dtype=float)

mean_sus = float(np.mean(participant_sus))
std_sus = float(np.std(participant_sus, ddof=1)) if N_PARTICIPANTS > 1 else 0.0
total_sum_all_participants = int(participant_sums.sum())

summary_with_stats = pd.concat(
    [
        scores_df,
        pd.DataFrame(
            [
                {
                    "Sum (0-40)": total_sum_all_participants,
                    "SUS score (0-100)": round(mean_sus, 2),
                },
                {
                    "Sum (0-40)": "-",
                    "SUS score (0-100)": round(std_sus, 2),
                },
            ],
            index=["Mean SUS Score", "Std. dev. (sample)"],
        ),
    ]
)

scores_path = EXPORT_DIR / "sus_scores_summary.csv"
summary_with_stats.reset_index(names="Participant").to_csv(scores_path, index=False)
print(f"Saved: {scores_path}")
print(f"Mean SUS (n={N_PARTICIPANTS}): {mean_sus:.2f}")
print(f"Std. dev. (sample): {std_sus:.2f}")
summary_with_stats

Saved: C:\Users\adria\OneDrive\Escriptori\UPF'\TFG\UPF_TFGRepository\notebooks\results visualizations\exports\sus_scores_summary.csv
Mean SUS (n=3): 70.83
Std. dev. (sample): 15.07


,Sum (0-40),SUS score (0-100)
Participant 1,22,55.00
Participant 2,29,72.50
Participant 3,34,85.00
Mean SUS Score,85,70.83
Std. dev. (sample),-,15.07
